In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

dense_model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True)


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [3]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

print("data loaded.")

data loaded.


In [4]:
import dense_index
from dense_index import DenseIndex

print(dense_model.normalize_embeddings)
court_dense_index = DenseIndex(dense_model, "../data/processed/_dense_sparse_court", court_doc)
court_dense_index.info()

True
DenseIndex.embeddings:  (2107648, 1024)
[dense_index] documents.len: 1985178 parent_idx.len: 2107648


In [5]:
def gen_train_top1000_dense():
    ret = []
    train_df = pd.read_csv("../data/train_rewrite_001.csv")
    for query in tqdm(train_df['query2'].tolist(), total=len(train_df)):
        ret.append(court_dense_index.search_with_score(query, 1000))
    return ret

In [6]:
TEMP_TRAIN_TOP100_DENSE = "../data/processed/train_top1000_dense.pkl"
if not os.path.exists(TEMP_TRAIN_TOP100_DENSE):
    l = gen_train_top1000_dense()
    with open(TEMP_TRAIN_TOP100_DENSE, "wb+") as of:
        pickle.dump(l, of)

train_top1000_dense_l = []

with open(TEMP_TRAIN_TOP100_DENSE, "rb") as inf:
    train_top1000_dense_l = pickle.load(inf)

In [7]:
def get_source(citation):
    if citation in court_consideration_d:
        return court_consideration_d[citation], True
    elif citation in law_d:
        return law_d[citation], False
    else:
        return None, False
        
def get_court(citation):
    if citation in court_consideration_d:
        return court_consideration_d[citation]
    else:
        return None

def extract_citations_from_text(text: str) -> list[str]:
    import re
    """Extract citations from any text (tool output or final answer)."""
    citations = []
    
    # SR pattern: SR followed by number (optionally with article)
    sr_matches = re.findall(
        r"SR\s*\d{3}(?:\.\d+)?(?:\s+Art\.?\s*\d+[a-z]?)?",
        text,
        re.IGNORECASE
    )
    citations.extend(sr_matches)
    
    # BGE pattern: BGE volume section page
    bge_matches = re.findall(
        r"BGE\s+\d{1,3}\s+[IVX]+[a-z]?\s+\d+(?:\s+E\.\s*\d+[a-z]?)?",
        text,
        re.IGNORECASE
    )
    citations.extend(bge_matches)
    
    # Art. pattern: Art. X LAW (e.g., Art. 1 ZGB, Art. 41 OR)
    art_matches = re.findall(
        r"Art\.?\s+\d+[a-z]?\s+(?:Abs\.?\s*\d+\s+)?[A-Z]{2,}",
        text,
        re.IGNORECASE
    )
    citations.extend(art_matches)
    
    return list(set(citations))

In [18]:
import citation_utils
import random

train_df = pd.read_csv("../data/train_rewrite_001.csv")

def gen_train_pos_neg(doc_score_l, gold_citations, query):
    gold_citation_set = set(gold_citations.split(';'))
    pos_d = dict()
    neg_d = dict()

    cited_gold_citation_set = set()
    for doc, score in doc_score_l:
        citation = doc['citation']
        text = get_court(citation)
        if text is None:
            continue

        e_c_l = extract_citations_from_text(text)
        for c in e_c_l:
            if c in gold_citation_set:
                cited_gold_citation_set.add(citation)

    
    pos_l = []

    # 生成正样本
    pos_neg_split_idx = -1
    for idx, (doc, score) in enumerate(doc_score_l):
        citation = doc['citation']
        if citation in cited_gold_citation_set:
            pos_l.append(doc)
            if idx >= 50:
                pos_neg_split_idx = idx
                break

    pos_neg_split_idx = max(pos_neg_split_idx, 50)

    hard_l = []
    for doc, score in doc_score_l[pos_neg_split_idx:pos_neg_split_idx+100]:
        citation = doc['citation']
        if citation in cited_gold_citation_set:
            continue 
        hard_l.append(doc)

    medium_l = []
    for doc, score in doc_score_l[pos_neg_split_idx+150:pos_neg_split_idx+300]:
        citation = doc['citation']
        if citation in cited_gold_citation_set:
            continue 
        medium_l.append(doc)

    random_l = []
    for doc, score in doc_score_l[pos_neg_split_idx+300:]:
        citation = doc['citation']
        if citation in cited_gold_citation_set:
            continue 
        random_l.append(doc)

    random.shuffle(hard_l)
    random.shuffle(medium_l)
    random.shuffle(random_l)
    
    return pos_l, hard_l[:len(pos_l)*2], medium_l[:len(pos_l)*3], random_l[:len(pos_l)*4]

     
    # print('p.len:', len(pos_l), 'h.len:', len(hard_l), 'm.len:', len(medium_l), 'r.len:',len(random_l))
        

In [16]:
# import citation_utils
# import random

# train_df = pd.read_csv("../data/train_rewrite_001.csv")

# def gen_train_pos_neg(doc_score_l, gold_citations, query):
#     gold_citation_set = set(gold_citations.split(';'))
#     pos_d = dict()
#     neg_d = dict()
#     # print("doc_score_l.len:", len(doc_score_l))

#     # for citation in gold_citation_set:
#     #     pos_d[citation] = 1


#     gold_citation_hit_count_d = {}

#     for doc, score in doc_score_l:
#         citation = doc['citation']
#         text, is_court = get_source(citation)
#         if text is None:
#             continue # citation is not in court considerations or law
#         if not is_court:
#             continue # is law

#         extract_citations = extract_citations_from_text(text)
#         for c in extract_citations:
#             if c in gold_citation_set:
#                 if c in gold_citation_hit_count_d:
#                     gold_citation_hit_count_d[c] += 1
#                 else:
#                     gold_citation_hit_count_d[c] = 1
    
#     for doc, score in doc_score_l:
#         citation = doc['citation']
#         text, is_court = get_source(citation)
#         if text is None:
#             continue # citation is not in court considerations or law
#         if not is_court:
#             continue # is law

#         extract_citations = extract_citations_from_text(text)
#         for c in extract_citations:
#             if c in gold_citation_set and gold_citation_hit_count_d.get(c, 0) > 2:
#                 pos_d[citation] = score
#             else:
#                 _text, _is_court = get_source(c)
#                 if _text is not None:
#                     if _is_court:
#                         neg_d[citation] = score

#     for pos,_ in pos_d.items():
#         if pos in neg_d:
#             del neg_d[pos]

#     for doc, score in doc_score_l:
#         citation = doc['citation']
#         text, is_court = get_source(citation)
#         if text is None:
#             continue # citation is not in court considerations or law
#         if not is_court:
#             continue # is law

#         extract_citations = extract_citations_from_text(text)
#         for c in extract_citations:
#             if c in gold_citation_set:
#                 if citation in neg_d:
#                     del neg_d[citation]

#     for doc, score in doc_score_l:
#         citation = doc['citation']
#         if citation not in pos_d:
#             neg_d[citation] = score
                    
#     ret_pos_l = []
#     ret_hard_neg_l = []
#     ret_medium_neg_l = []
#     ret_random_neg_l = []
    
#     for citation, score in pos_d.items():            
#         passage, _ = get_source(citation)
#         ret_pos_l.append({'query':query, 'passage':passage, 'label':1})

#     neg_l = sorted([(citation, score) for citation, score in neg_d.items()], key=lambda x: x[1], reverse=True)

#     pos_count = len(ret_pos_l)

#     print("pos.len:", len(ret_pos_l), "neg.len:", len(neg_l))

#     _hard_l = [item for item in neg_l[10:50]]
#     random.shuffle(_hard_l)
#     hard_neg_l = _hard_l[:pos_count*2]
#     for citation, _ in hard_neg_l:
#         passage, _ = get_source(citation)
#         ret_hard_neg_l.append({'query':query, 'passage':passage, 'label':0, 'type':'hard'})

#     _medium_l = [item for item in neg_l[50:150]]
#     random.shuffle(_medium_l)
#     medium_neg_l = _medium_l[:pos_count*4]
#     for citation, _ in medium_neg_l:
#         passage, _ = get_source(citation)
#         ret_medium_neg_l.append({'query':query, 'passage':passage, 'label':0, 'type':'medium'})

#     _random_l = [item for item in neg_l[150:]]
#     random.shuffle(_random_l)
#     random_neg_l = _random_l[:pos_count*6]
#     for citation, _ in random_neg_l:
#         passage, _ = get_source(citation)
#         ret_random_neg_l.append({'query':query, 'passage':passage, 'label':0, 'type':'random'})

#     print("pos.len:", len(ret_pos_l), "hard.len:", len(ret_hard_neg_l), 'medium.len:', len(ret_medium_neg_l), 'random.len:', len(ret_random_neg_l))

#     return ret_pos_l, ret_hard_neg_l, ret_medium_neg_l, ret_random_neg_l

In [27]:
import json

def _to_dict(query, passage, label, type):
    if label == 0:
        return {'query':query, 'passage':passage, 'label':label, 'type':type}
    else:
        return {'query':query, 'passage':passage, 'label':label}

of = open("../ft_data/train3.jsonl", "w+", encoding="utf-8")

result_l = []
for doc_score_l, gold_citations, query in tqdm(zip(train_top1000_dense_l, 
                                       train_df['gold_citations'].tolist(),
                                       train_df['query2'].tolist()), total=len(train_df)):
    pos_l, hard_neg_l, medium_neg_l, rand_neg_l = gen_train_pos_neg(doc_score_l, gold_citations, query)

    for pos in pos_l:
        d = _to_dict(query, pos['text'], 1, None)
        json_line = json.dumps(d, ensure_ascii=False)
        of.write(json_line + '\n')
    for neg in hard_neg_l:
        d = _to_dict(query, neg['text'], 0, 'hard')
        json_line = json.dumps(d, ensure_ascii=False)
        of.write(json_line + '\n')
    for neg in medium_neg_l:
        d = _to_dict(query, neg['text'], 0, 'medium')
        json_line = json.dumps(d, ensure_ascii=False)
        of.write(json_line + '\n')
    for neg in rand_neg_l:
        d = _to_dict(query, neg['text'], 0, 'random')
        json_line = json.dumps(d, ensure_ascii=False)
        of.write(json_line + '\n')

of.close()

100%|██████████| 1139/1139 [01:17<00:00, 14.67it/s]


In [28]:
import json
import random

sample_l = []
with open("../ft_data/train3.jsonl", 'r', encoding='utf-8') as inf:
    for line in inf:
        sample_l.append(json.loads(line.strip()))

random.seed(42)

print("sample_l.len:", len(sample_l))

random.shuffle(sample_l)

train_sample_l, valid_sample_l = sample_l[:60000], sample_l[60000:]
print(f"{len(train_sample_l)}, {len(valid_sample_l)}")
with open("../ft_data/train3_train.jsonl", 'w', encoding='utf-8') as f:
    for d in train_sample_l:
        json_line = json.dumps(d, ensure_ascii=False)
        f.write(json_line + '\n')
        
with open("../ft_data/train3_valid.jsonl", 'w', encoding='utf-8') as f:
    for d in valid_sample_l:
        json_line = json.dumps(d, ensure_ascii=False)
        f.write(json_line + '\n')

sample_l.len: 64398
60000, 4398
